# Approche C — STUNet + Velocity-Guided Timing-SAM
**Sara** : T5 encoder + diffusion DDIM + loss  
**Hiba** : STUNet + Timing-SAM + velocity injection

**Dataset :** How2Sign — 151 keypoints, séquences jusqu'à 440 frames

## 1. Setup — clone repo + imports

In [1]:
!git clone https://github.com/sarrazer24/sign-language-production.git
!pip install transformers sentencepiece dtaidistance -q

Cloning into 'sign-language-production'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 120 (delta 65), reused 62 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 250.09 KiB | 3.73 MiB/s, done.
Resolving deltas: 100% (65/65), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 44.6 MB/s eta 0:00:00a 0:00:01


In [4]:
import sys
sys.path.append('/kaggle/working/sign-language-production/phase1_text_to_pose')

import os, time, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from data.dataset  import How2SignDataset
from data.collate  import collate_fn
from models.approach_c.stunet_timingsam import (
    SignSAM_C, GaussianDiffusion, reconstruction_loss, VelocityGuidedTimingSAM
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

Device : cuda
GPU    : Tesla T4


## 2. Dataset + DataLoader (bloc commun des instructions)

In [5]:
stats = torch.load('/kaggle/working/sign-language-production/phase1_text_to_pose/data/stats.pt')

train_ds = How2SignDataset('train', stats=stats, max_frames=500)
dev_ds   = How2SignDataset('dev',   stats=stats, max_frames=500)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_ds,   batch_size=32, shuffle=False, collate_fn=collate_fn)

# Vérification rapide (bloc imposé par les instructions)
batch = next(iter(train_loader))
print(batch['poses'].shape)    # torch.Size([32, T, 151, 3])
print(batch['pose_mask'].shape) # torch.Size([32, T])
print(batch['texts'][0])        # une phrase en anglais

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

torch.Size([32, 178, 151, 3])
torch.Size([32, 178])
"Once you have it squeegeed, you can simply take your masking slowly peeling it way, keeping it as close to the masking as possible."


## 3. Config

In [6]:
N_KEYPOINTS     = 151
MODEL_CHANNELS  = 256
NUM_HEADS       = 4
DROPOUT         = 0.1
DIFFUSION_STEPS = 50      # DDIM, instructions Approche C
GUIDANCE_SCALE  = 5.5
LEARNING_RATE   = 1e-4
NUM_EPOCHS      = 50
LR_GAMMA        = 0.99998
SAVE_EVERY      = 5

SAVE_DIR = '/kaggle/working/checkpoints_c'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Config OK')

Config OK


## 4. Modèle + Diffusion

In [ ]:
model     = SignSAM_C(n_keypoints=N_KEYPOINTS, model_channels=MODEL_CHANNELS, dropout=DROPOUT).to(DEVICE)
diffusion = GaussianDiffusion(T=DIFFUSION_STEPS, device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=LR_GAMMA)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres total       : {total:,}')
print(f'Paramètres entraînables: {trainable:,}')

## 5. Fonctions train / validate

In [ ]:
def train_one_epoch(model, loader, optimizer, diffusion, device):
    model.train()
    total, n = 0.0, 0

    for batch in loader:
        poses = batch['poses'].to(device)                  # (B, T, 151, 3)
        mask  = batch['pose_mask'].float().to(device)      # (B, T)
        ids   = batch['input_ids'].to(device)              # (B, L)
        amask = batch['attention_mask'].to(device)         # (B, L)

        B = poses.shape[0]
        t = torch.randint(0, diffusion.T, (B,), device=device)

        # Forward diffusion : ajouter du bruit
        x_noisy = diffusion.q_sample(poses, t)

        # Prédire x̂_0
        optimizer.zero_grad()
        x_pred = model(x_noisy, t, ids, amask)

        # Loss Smooth-L1 poses + vélocités
        loss = reconstruction_loss(x_pred, poses, mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total += loss.item()
        n     += 1

    return total / n


@torch.no_grad()
def validate(model, loader, diffusion, device):
    model.eval()
    total, n = 0.0, 0

    for batch in loader:
        poses = batch['poses'].to(device)
        mask  = batch['pose_mask'].float().to(device)
        ids   = batch['input_ids'].to(device)
        amask = batch['attention_mask'].to(device)

        B = poses.shape[0]
        t = torch.randint(0, diffusion.T, (B,), device=device)
        x_noisy = diffusion.q_sample(poses, t)
        x_pred  = model(x_noisy, t, ids, amask)
        loss    = reconstruction_loss(x_pred, poses, mask)

        total += loss.item()
        n     += 1

    return total / n

## 6. Boucle d'entraînement

In [ ]:
# Reprendre depuis un checkpoint si disponible
start_epoch  = 0
train_losses = []
val_losses   = []
best_val     = float('inf')

ckpt_path = os.path.join(SAVE_DIR, 'latest.pt')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch  = ckpt['epoch'] + 1
    train_losses = ckpt.get('train_losses', [])
    val_losses   = ckpt.get('val_losses',   [])
    best_val     = ckpt.get('best_val',     float('inf'))
    print(f'Reprise à l\'epoch {start_epoch}')
else:
    print('Démarrage from scratch.')

In [ ]:
print(f'Entraînement Approche C — {NUM_EPOCHS} epochs sur {DEVICE}\n')

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()

    tl = train_one_epoch(model, train_loader, optimizer, diffusion, DEVICE)
    vl = validate(model, dev_loader, diffusion, DEVICE)
    scheduler.step()

    train_losses.append(tl)
    val_losses.append(vl)

    print(f'Epoch {epoch+1:03d}/{NUM_EPOCHS}  '
          f'train={tl:.4f}  val={vl:.4f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  '
          f'time={time.time()-t0:.1f}s')

    # Sauvegarder le meilleur modèle sur Google Drive
    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'best.pt'))
        print(f'  ✓ Best model (val={vl:.4f})')

    # Checkpoint périodique
    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'train_losses': train_losses,
            'val_losses': val_losses,
            'best_val': best_val,
        }, ckpt_path)
        print(f'  Checkpoint sauvegardé.')

print('\nEntraînement terminé!')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Approche C — Courbe d\'entraînement')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'loss_curve.png'), dpi=150)
plt.show()

## 7. Ablation — avec vs sans velocity injection
Requis par les instructions. On compare :
- **Baseline** : BiGRU reçoit `X` seulement
- **Ours** : BiGRU reçoit `[X, Vel(X)]`

In [ ]:
# ── Modèle SANS vélocité (baseline pour ablation) ─────────────────────────────
_orig_vel = VelocityGuidedTimingSAM.compute_velocity
VelocityGuidedTimingSAM.compute_velocity = staticmethod(lambda x: torch.zeros_like(x))
model_base = SignSAM_C(n_keypoints=N_KEYPOINTS, model_channels=MODEL_CHANNELS).to(DEVICE)
VelocityGuidedTimingSAM.compute_velocity = staticmethod(_orig_vel)  # restaurer

# ── Modèle AVEC vélocité (ours) ───────────────────────────────────────────────
model_vel = SignSAM_C(n_keypoints=N_KEYPOINTS, model_channels=MODEL_CHANNELS).to(DEVICE)

opt_base = torch.optim.AdamW(model_base.parameters(), lr=LEARNING_RATE)
opt_vel  = torch.optim.AdamW(model_vel.parameters(),  lr=LEARNING_RATE)

ABLATION_EPOCHS = 10
abl = {'base': {'train': [], 'val': []}, 'vel': {'train': [], 'val': []}}

for ep in range(ABLATION_EPOCHS):
    tl_b = train_one_epoch(model_base, train_loader, opt_base, diffusion, DEVICE)
    vl_b = validate(model_base, dev_loader, diffusion, DEVICE)
    abl['base']['train'].append(tl_b)
    abl['base']['val'].append(vl_b)

    tl_v = train_one_epoch(model_vel, train_loader, opt_vel, diffusion, DEVICE)
    vl_v = validate(model_vel, dev_loader, diffusion, DEVICE)
    abl['vel']['train'].append(tl_v)
    abl['vel']['val'].append(vl_v)

    print(f'Ep {ep+1:02d}  Baseline={vl_b:.4f}  Velocity-Guided={vl_v:.4f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key in zip(axes, ['train', 'val']):
    ax.plot(abl['base'][key], '--', label='Baseline (sans vélocité)')
    ax.plot(abl['vel'][key],        label='Velocity-Guided (ours)')
    ax.set_title(f'{key.capitalize()} loss')
    ax.legend()
plt.suptitle('Ablation : Velocity-Guided Timing-SAM')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'ablation.png'), dpi=150)
plt.show()

print(f"\nVal finale — Baseline       : {abl['base']['val'][-1]:.4f}")
print(f"Val finale — Velocity-Guided: {abl['vel']['val'][-1]:.4f}")

In [1]:
!git clone https://github.com/sarrazer24/sign-language-production.git
%cd sign-language-production
!git pull origin main

# 3. Installer les dépendances
!pip install transformers sentencepiece -q

import sys
sys.path.append('/kaggle/working/sign-language-production/phase1_text_to_pose')

import torch
import numpy as np
import matplotlib.pyplot as plt

from models.approach_c.stunet_timingsam import SignSAM_C, GaussianDiffusion

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

Cloning into 'sign-language-production'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 125 (delta 68), reused 60 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 2.32 MiB | 8.72 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/kaggle/working/sign-language-production
From https://github.com/sarrazer24/sign-language-production
 * branch            main       -> FETCH_HEAD
Already up to date.
Device : cuda


In [12]:
stats = torch.load('/kaggle/working/sign-language-production/phase1_text_to_pose/data/stats.pt')
print('Stats chargées')

Stats chargées


In [13]:
# ── Charger le modèle ──────────────────────────────────────────────────────
from transformers import T5Tokenizer

model_infer = SignSAM_C(n_keypoints=151, model_channels=256, dropout=0.0).to(DEVICE)
model_infer.load_state_dict(torch.load('/kaggle/input/models/sidsara/best/pytorch/default/1/best.pt', map_location=DEVICE), strict=False)
model_infer.eval()
print('✓ Modèle chargé')

tokenizer = T5Tokenizer.from_pretrained('t5-small')
diffusion_infer = GaussianDiffusion(T=50, device=DEVICE)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

✓ Modèle chargé


In [16]:
# ── Fonction de prédiction ─────────────────────────────────────────────────
def predict_poses(phrase: str, n_frames: int = 60):
    """
    phrase   : une phrase en anglais
    n_frames : nombre de frames à générer (60 ≈ 2 secondes)
    Retourne : poses (n_frames, 151, 3) — numpy array
    """
    # 1. Tokeniser le texte
    enc = tokenizer(
        phrase,
        return_tensors  = 'pt',
        padding         = True,
        truncation      = True,
        max_length      = 200,
    )
    ids   = enc['input_ids'].to(DEVICE)       # (1, L)
    amask = enc['attention_mask'].to(DEVICE)  # (1, L)

    # 2. Générer les poses avec DDIM
    with torch.no_grad():
        poses = diffusion_infer.ddim_sample(
            model_infer,
            input_ids      = ids,
            attention_mask = amask,
            shape          = (1, n_frames, 151, 3),
            guidance_scale = 5.5,
            device         = DEVICE,
        )   # (1, n_frames, 151, 3)

    # 3. Dénormaliser (remettre dans les vraies unités)
    poses_np = poses[0].cpu().numpy()                    # (n_frames, 151, 3)
    mean     = stats['mean'].numpy()                     # (151, 3)
    std      = stats['std'].numpy()                      # (151, 3)
    poses_np = poses_np * std + mean                     # dénormalisation

    return poses_np

In [17]:
# Créer le test loader
from data.dataset import How2SignDataset
from data.collate import collate_fn
from torch.utils.data import DataLoader

test_ds     = How2SignDataset('test', stats=stats, max_frames=500)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)

print(f'Test set : {len(test_ds)} samples')

Test set : 2343 samples


In [9]:
# ── Évaluation complète sur le test set ───────────────────────────────────
!pip install dtaidistance -q

from dtaidistance import dtw_ndim

# ── Métriques ──────────────────────────────────────────────────────────────
def mpjpe(pred, target):
    return float(np.mean(np.linalg.norm(pred - target, axis=-1)))

def dtw_distance(pred, target):
    try:
        d = dtw_ndim.distance(pred.astype(np.double), target.astype(np.double))
        return float(d / (len(pred) + len(target)))
    except:
        return float(np.mean(np.abs(pred - target)))

def velocity_error(pred, target):
    vel_pred   = pred[1:] - pred[:-1]
    vel_target = target[1:] - target[:-1]
    return float(np.mean(np.linalg.norm(vel_pred - vel_target, axis=-1)))

# ── Charger le meilleur modèle ─────────────────────────────────────────────
model_eval = SignSAM_C(
    n_keypoints    = N_KEYPOINTS,
    model_channels = MODEL_CHANNELS,
    dropout        = 0.0,
).to(DEVICE)

model_eval.load_state_dict(
    torch.load(os.path.join(SAVE_DIR, '/kaggle/input/models/sidsara/best/pytorch/default/1/best.pt'), map_location=DEVICE)
)
model_eval.eval()
print('✓ Meilleur modèle chargé')

diffusion_eval = GaussianDiffusion(T=DIFFUSION_STEPS, device=DEVICE)

# ── Test loader ────────────────────────────────────────────────────────────
test_ds     = How2SignDataset('test', stats=stats, max_frames=500)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False,
                         collate_fn=collate_fn, num_workers=2)
print(f'Test set : {len(test_ds)} samples')

# ── Boucle d'évaluation ────────────────────────────────────────────────────
all_mpjpe = []
all_dtw   = []
all_vel   = []

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        poses   = batch['poses'].to(DEVICE)
        ids     = batch['input_ids'].to(DEVICE)
        amask   = batch['attention_mask'].to(DEVICE)
        lengths = batch['pose_lengths'].tolist()

        B, T, K, _ = poses.shape

        preds = diffusion_eval.ddim_sample(
            model_eval, ids, amask,
            shape          = (B, T, K, 3),
            guidance_scale = GUIDANCE_SCALE,
            device         = DEVICE,
        )

        for j, length in enumerate(lengths):
            # Ignorer séquences trop courtes ou vides
            if length < 2:
                continue

            gt   = poses[j, :length].cpu().numpy()
            pred = preds[j, :length].cpu().numpy()

            if gt.shape[0] == 0 or pred.shape[0] == 0:
                continue

            all_mpjpe.append(mpjpe(pred, gt))
            all_dtw.append(dtw_distance(
                pred.reshape(length, -1),
                gt.reshape(length, -1)
            ))
            all_vel.append(velocity_error(pred, gt))

        if (i + 1) % 10 == 0:
            print(f'Batch {i+1}/{len(test_loader)}  '
                  f'MPJPE={np.mean(all_mpjpe):.4f}  '
                  f'DTW={np.mean(all_dtw):.4f}  '
                  f'Vel={np.mean(all_vel):.4f}')

# ── Résultats finaux ───────────────────────────────────────────────────────
print('\n========== RÉSULTATS FINAUX ==========')
print(f'MPJPE          : {np.mean(all_mpjpe):.4f} ± {np.std(all_mpjpe):.4f}')
print(f'DTW            : {np.mean(all_dtw):.4f} ± {np.std(all_dtw):.4f}')
print(f'Velocity Error : {np.mean(all_vel):.4f} ± {np.std(all_vel):.4f}')

# ── Tableau ────────────────────────────────────────────────────────────────
import pandas as pd
df = pd.DataFrame({
    'Métrique' : ['MPJPE', 'DTW', 'Velocity Error'],
    'Moyenne'  : [np.mean(all_mpjpe), np.mean(all_dtw), np.mean(all_vel)],
    'Std'      : [np.std(all_mpjpe),  np.std(all_dtw),  np.std(all_vel)],
})
print('\n', df.to_string(index=False))
df.to_csv(os.path.join(SAVE_DIR, 'metrics.csv'), index=False)
print('\n✓ Résultats sauvegardés dans metrics.csv')

Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

✓ Meilleur modèle chargé
Test set : 2343 samples
Batch 10/293  MPJPE=1.9451  DTW=2.1621  Vel=0.5553
Batch 20/293  MPJPE=1.9552  DTW=2.1644  Vel=0.6201
Batch 30/293  MPJPE=1.9688  DTW=2.1953  Vel=0.6882
Batch 40/293  MPJPE=1.9756  DTW=2.2483  Vel=0.7093
Batch 50/293  MPJPE=1.9797  DTW=2.2457  Vel=0.7063
Batch 60/293  MPJPE=2.0039  DTW=2.3319  Vel=0.7002
Batch 70/293  MPJPE=2.0245  DTW=2.3968  Vel=0.7029
Batch 80/293  MPJPE=2.0086  DTW=2.3343  Vel=0.6998
Batch 90/293  MPJPE=1.9759  DTW=2.2658  Vel=0.6995
Batch 100/293  MPJPE=1.9696  DTW=2.2640  Vel=0.6960
Batch 110/293  MPJPE=1.9481  DTW=2.2223  Vel=0.6952
Batch 120/293  MPJPE=1.9550  DTW=2.2176  Vel=0.7003
Batch 130/293  MPJPE=1.9544  DTW=2.1763  Vel=0.7058
Batch 140/293  MPJPE=1.9537  DTW=2.2214  Vel=0.7036
Batch 150/293  MPJPE=1.9607  DTW=2.2306  Vel=0.6991
Batch 160/293  MPJPE=1.9629  DTW=2.2255  Vel=0.7002
Batch 170/293  MPJPE=1.9767  DTW=2.2911  Vel=0.7123
Batch 180/293  MPJPE=1.9911  DTW=2.3467  Vel=0.7174
Batch 190/293  MPJPE=1.9